<a href="https://colab.research.google.com/github/szymonbloch/tuberculosis_detection/blob/main/preparing_and_testing_extended_focus_layer_3_layers_with_colors.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Data preparing

In [ ]:
import pandas as pd
import glob
import os
import numpy as np
import joblib
from sklearn.metrics import balanced_accuracy_score, confusion_matrix

# ==========================================
# 1. WCZYTANIE ETYKIET Z PLIKÓW TXT
# ==========================================
POS_TXT_PATH = '/content/drive/MyDrive/NTWI_project/data/positive.txt'
NEG_TXT_PATH = '/content/drive/MyDrive/NTWI_project/data/negative.txt'

with open(POS_TXT_PATH, 'r') as f:
    pos_slides = [line.strip() for line in f if line.strip()]

with open(NEG_TXT_PATH, 'r') as f:
    neg_slides = [line.strip() for line in f if line.strip()]

# Budujemy słownik mapujący: {nazwa_slajdu: 0 lub 1}
labels_dict = {}
for slide in pos_slides:
    labels_dict[slide] = 1
for slide in neg_slides:
    labels_dict[slide] = 0

df_final_3l = pd.DataFrame(list(labels_dict.items()), columns=['image_name', 'image_output'])

In [ ]:
# ==========================================
# 2. DETEKTOR ANOMALII (TYLKO WARSTWA EXTENDED)
# ==========================================
ANOMALY_3L_PATH = '/content/drive/MyDrive/NTWI_project/data/anomaly_detector/3-layer_ad_cnn_04_06_2025_00_31_21/*.csv'
anomaly_files = glob.glob(ANOMALY_3L_PATH)
table_list_anomaly_ext = []

for file in anomaly_files:
    basename = os.path.basename(file)
    # Wyłapujemy tylko i wyłącznie pliki z końcówką Extended.csv
    if '_Wholeslide_Default_Extended.csv' in basename:
        slide_name = basename.split('_Wholeslide_Default_')[0]

        df_temp = pd.read_csv(file)
        df_temp['probability'] = df_temp['probability'].astype(str).str.extract(r'tensor\(([0-9.]+)').astype(float)

        df_temp['tile_coord'] = df_temp['image_name'].astype(str).str.split('.').str[0].str.strip()
        df_temp['tile_name'] = slide_name + '_' + df_temp['tile_coord']
        df_temp['image_name'] = slide_name

        df_temp = df_temp[['tile_name', 'image_name', 'probability', 'output']]
        table_list_anomaly_ext.append(df_temp)

df_anomaly_ext = pd.concat(table_list_anomaly_ext, ignore_index=True)
df_anomaly_ext.rename(columns={'output': 'tile_output'}, inplace=True)


In [ ]:
# ==========================================
# 3. CECHY KOLORYSTYCZNE (TYLKO WARSTWA EXTENDED)
# ==========================================
COLORS_3L_PATH = '/content/drive/MyDrive/NTWI_project/data/color_features/3_layers/*.csv'
color_files = glob.glob(COLORS_3L_PATH)
table_list_colors_ext = []

for file in color_files:
    basename = os.path.basename(file)

    if '_Wholeslide_Default_Extended.csv' in basename:
        slide_name = basename.split('_Wholeslide_Default_')[0]

        df_temp = pd.read_csv(file, header=None, sep=None, engine='python')
        df_temp.rename(columns={0: 'tile_coord'}, inplace=True)
        df_temp['tile_coord'] = df_temp['tile_coord'].astype(str).str.split('.').str[0].str.strip()

        df_temp['tile_name'] = slide_name + '_' + df_temp['tile_coord']
        df_temp.drop(columns=['tile_coord'], inplace=True)

        table_list_colors_ext.append(df_temp)

df_colors_ext_raw = pd.concat(table_list_colors_ext, ignore_index=True)

# USUWANIE ZŁOŚLIWEJ KOLUMNY 59
df_colors_ext_raw = df_colors_ext_raw.dropna(axis=1, how='all')
if 59 in df_colors_ext_raw.columns:
    df_colors_ext_raw = df_colors_ext_raw.drop(columns=[59])
if '59' in df_colors_ext_raw.columns:
    df_colors_ext_raw = df_colors_ext_raw.drop(columns=['59'])

df_colors_ext = df_colors_ext_raw


# Data merging

In [ ]:
# ==========================================
# 4. WIELKI MERGE DANYCH EXTENDED
# ==========================================
df_step1_ext = df_anomaly_ext.merge(df_final_3l, on='image_name', how='inner')
df_merged_ext = df_step1_ext.merge(df_colors_ext, on='tile_name', how='inner')
df_merged_ext.to_csv('/content/drive/MyDrive/NTWI_project/results/df_merged_extened_3l.csv', index=False)


# Statistics from merged data

In [ ]:
import pandas as pd

MERGED_EXTENDED_PATH = '/content/drive/MyDrive/NTWI_project/results/df_merged_extended_3l.csv'
STATISTICS_EXTENDED_PATH = '/content/drive/MyDrive/NTWI_project/results/df_statistics_extended_3l.csv'

# 1. Wczytanie pliku
df_extended = pd.read_csv(MERGED_EXTENDED_PATH)

# ZABEZPIECZENIE NR 2: Ujednolicenie nazwy diagnozy
if 'image_output' in df_extended.columns and 'output_image' not in df_extended.columns:
    df_extended = df_extended.rename(columns={'image_output': 'output_image'})

# 2. Przygotowanie słownika statystyk
statistic_dict = {
    'probability': ['mean', 'max', 'std'],
    'output_image': ['first']
}

# Dodanie średniej dla wszystkich cech kolorystycznych (kolumny z nazwami liczbowymi)
for col in df_extended.columns:
    if str(col).isdigit():
        statistic_dict[col] = ['mean']

# 3. Agregacja pacjentów (Global Statistical Pooling)
df_ext_stats = df_extended.groupby('image_name').agg(statistic_dict).reset_index()

# Spłaszczanie nazw kolumn (np. probability -> mean zamienia się w probability_mean)
df_ext_stats.columns = ['_'.join([str(c) for c in col]).strip('_') for col in df_ext_stats.columns.values]
df_ext_stats.rename(columns={'output_image_first': 'output_image'}, inplace=True)

# 4. Zapis sterylnie czystego pliku (bez nowych śmieciowych indeksów)
df_ext_stats.to_csv(STATISTICS_EXTENDED_PATH, index=False)

print(f"Czysty plik statystyk dla warstwy Extended zapisany w:\n{STATISTICS_EXTENDED_PATH}")
print(f"Kształt tabeli: {df_ext_stats.shape} (Pacjenci x Cechy)")

# ML model testing (Random Forest)

In [ ]:
import pandas as pd
import joblib
from sklearn.metrics import balanced_accuracy_score, confusion_matrix

# ==========================================
# ŚCIEŻKI DO PLIKÓW
# ==========================================
STATISTICS_EXTENDED_PATH = '/content/drive/MyDrive/NTWI_project/results/df_statistics_extended_3l.csv'
MODEL_PATH = '/content/drive/MyDrive/NTWI_project/models/statistics_rf.pkl'

# ==========================================
# 1. WCZYTANIE DANYCH EXTENDED
# ==========================================
df_ext_stats = pd.read_csv(STATISTICS_EXTENDED_PATH)

# Przygotowanie cech (X) i etykiet (y)
X_test_ext = df_ext_stats.drop(columns=['image_name', 'output_image'])
y_test_ext = df_ext_stats['output_image']

# ==========================================
# 2. WCZYTANIE MODELU I DOPASOWANIE CECH
# ==========================================
rf_model_loaded = joblib.load(MODEL_PATH)

# Wymuszenie identycznej kolejności i nazw kolumn, jak podczas treningu
if hasattr(rf_model_loaded, "feature_names_in_"):
    X_test_ext = X_test_ext[rf_model_loaded.feature_names_in_]

# ==========================================
# 3. TEST I WYNIKI
# ==========================================
y_pred_ext = rf_model_loaded.predict(X_test_ext)
bacc_ext = balanced_accuracy_score(y_test_ext, y_pred_ext)

print(f"\nOstateczny wynik na warstwie Extended (Statystyki + Kolory): {bacc_ext * 100:.2f}%")
print("Macierz pomyłek dla warstwy Extended:\n", confusion_matrix(y_test_ext, y_pred_ext))

# Multiple Instance Learning (top-K) testing

In [ ]:
# ==========================================
# 5. AGREGACJA TOP-50 DO POZIOMU PACJENTA
# ==========================================

MERGED_DATAFRAME_PATH = '/content/drive/MyDrive/NTWI_project/results/df_merged_extended_3l.csv'
df_merged_ext = pd.read_csv(MERGED_DATAFRAME_PATH)

K = 50
top_k_bags_ext = []
labels_ext = []
slide_names_ext = []

grouped = df_merged_ext.groupby('image_name')

for name, group in grouped:
    group_sorted = group.sort_values(by='probability', ascending=False)
    top_k_tiles = group_sorted.head(K)

    label = top_k_tiles['image_output'].iloc[0]
    labels_ext.append(label)
    slide_names_ext.append(name)

    features_cols = ['probability'] + [col for col in group.columns if str(col).isdigit()]
    flat_features = top_k_tiles[features_cols].values.flatten()

    if len(flat_features) < K * len(features_cols):
        padding = np.zeros(K * len(features_cols) - len(flat_features))
        flat_features = np.concatenate((flat_features, padding))

    top_k_bags_ext.append(flat_features)

X_test_ext = np.array(top_k_bags_ext)
y_test_ext = np.array(labels_ext)

In [ ]:
# ==========================================
# 6. POBRANIE WYTRENOWANEGO MODELU I TEST
# ==========================================
rf_model_loaded = joblib.load('/content/drive/MyDrive/NTWI_project/models/best_k5_rf.pkl')

y_pred_ext = rf_model_loaded.predict(X_test_ext)
bacc_ext = balanced_accuracy_score(y_test_ext, y_pred_ext)

print(f"\nOstateczny wynik Balanced Accuracy na warstwie Extended: {bacc_ext * 100:.2f}%")
print("Macierz pomyłek dla warstwy Extended:\n", confusion_matrix(y_test_ext, y_pred_ext))


Ostateczny wynik Balanced Accuracy na warstwie Extended: 43.33%
Macierz pomyłek dla warstwy Extended:
 [[2 3]
 [8 7]]
